# LeetCode 783 — Minimum Distance Between BST Nodes bằng Python

**Link đề:** https://leetcode.com/problems/minimum-distance-between-bst-nodes/
**Độ khó:** Easy · **Chủ đề:** Tree, DFS, BFS, Binary Search Tree (BST), Binary Tree
*(Bài này giống hệt LeetCode 530 — Minimum Absolute Difference in BST.)*

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def minDiffInBST(self, root: TreeNode | None) -> int:
        ...
```

## Mục tiêu học
1. Tự định nghĩa `TreeNode`, dựng cây từ list kiểu LeetCode `[4,2,6,1,3]` và in cây để quan sát.
2. Hiểu **BST** (cây nhị phân tìm kiếm) và tính chất vàng: **duyệt inorder cho dãy tăng dần**.
3. Đi từ brute force mọi cặp `O(n²)` → sắp xếp `O(n log n)` → inorder + so phần tử liền kề `O(n)`.
4. Viết inorder **đệ quy** (dùng `nonlocal prev`) và **lặp bằng stack**.
5. Trace, chứng minh "min chỉ nằm ở cặp liền kề", phân tích Time/Space.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho `root` của một **cây nhị phân tìm kiếm (BST)**. Trả về **hiệu nhỏ nhất** giữa giá trị của **hai nút khác nhau bất kỳ** trong cây.

**Ví dụ 1:** `root = [4,2,6,1,3]` → `1`
```text
        4
      /   \
     2     6
    / \
   1   3
```
Các giá trị: 1, 2, 3, 4, 6 → hiệu nhỏ nhất là 1 (ví dụ 2−1).

**Ví dụ 2:** `root = [1,0,48,null,null,12,49]` → `1`
```text
      1
     / \
    0   48
       /  \
      12   49
```
Giá trị: 0, 1, 12, 48, 49 → hiệu nhỏ nhất 1 (1−0 hoặc 49−48).

**Constraints:** số nút trong `[2, 100]`; `0 <= Node.val <= 10^5`.

Lưu ý: "hai nút **bất kỳ**", không nhất thiết là cha–con.

## 2. Nền tảng: class `TreeNode` và dựng cây từ list LeetCode

Mỗi nút là một **object** có 3 thuộc tính: `val`, `left`, `right`. `None` nghĩa là "không có con".

LeetCode biểu diễn cây theo **thứ tự từng tầng** (level order), `None`/`null` đánh dấu chỗ trống. Ta viết hàm `build_tree` dùng **hàng đợi** (`deque`) để gắn con lần lượt, và `tree_to_list` để chuyển ngược lại kiểm tra.

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(values):
    # values: list kiểu LeetCode, ví dụ [4, 2, 6, 1, 3] hoặc [1, 0, 48, None, None, 12, 49]
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:      # bỏ các None thừa ở cuối
        out.pop()
    return out

root1 = build_tree([4, 2, 6, 1, 3])
root2 = build_tree([1, 0, 48, None, None, 12, 49])
print(root1, root1.left, root1.right, root1.left.left, root1.left.right)
print(tree_to_list(root1))
print(tree_to_list(root2))

TreeNode(4) TreeNode(2) TreeNode(6) TreeNode(1) TreeNode(3)
[4, 2, 6, 1, 3]
[1, 0, 48, None, None, 12, 49]


### 2.1. `deque` — hàng đợi hai đầu

`deque` (module `collections`) cho phép `append` vào cuối và `popleft` lấy ở đầu đều `O(1)` (list `pop(0)` tốn `O(n)`). Hàng đợi = "vào trước ra trước" (FIFO), đúng thứ tự từng tầng.

In [2]:
q = deque([1, 2])
q.append(3)
print(q, "| popleft ->", q.popleft(), "| còn", q)

deque([2, 3]) | popleft -> 1 | còn deque([2, 3])


### 2.2. In cây nằm ngang để quan sát

Hàm `print_tree` in cây xoay 90° (con phải ở trên, con trái ở dưới), dùng **đệ quy**: một hàm gọi lại chính nó cho cây con. Điều kiện dừng: `node is None`.

In [3]:
def print_tree(node, depth=0):
    if node is None:
        return
    print_tree(node.right, depth + 1)
    print("    " * depth + str(node.val))
    print_tree(node.left, depth + 1)

print_tree(root1)
print("-----")
print_tree(root2)

    6
4
        3
    2
        1
-----
        49
    48
        12
1
    0


## 3. Nền tảng: BST là gì?

**Binary Search Tree**: với **mọi** nút `x`:
- mọi giá trị trong **cây con trái** `< x.val`,
- mọi giá trị trong **cây con phải** `> x.val`.

(Trong bài này các giá trị khác nhau; nếu có trùng, đáp án là 0 và thuật toán vẫn đúng.)

### Duyệt inorder: trái → gốc → phải
```python
def inorder(node):
    if node is None: return
    inorder(node.left)
    thăm(node)
    inorder(node.right)
```
Vì trái < gốc < phải ở **mọi** nút, inorder trên BST cho ra dãy **tăng dần**. Đây là chìa khóa của bài.

In [4]:
def inorder_values(node, out=None):
    if out is None:
        out = []
    if node is not None:
        inorder_values(node.left, out)
        out.append(node.val)
        inorder_values(node.right, out)
    return out

print("inorder root1:", inorder_values(root1))
print("inorder root2:", inorder_values(root2))

inorder root1: [1, 2, 3, 4, 6]
inorder root2: [0, 1, 12, 48, 49]


**Vì sao `out=None` rồi mới tạo `[]`?** Nếu viết `def f(node, out=[])`, list mặc định được tạo **một lần** khi định nghĩa hàm và bị dùng chung giữa các lần gọi → dữ liệu lần trước "rò" sang lần sau. Đây là bẫy kinh điển của Python.

In [5]:
def bad_append(x, acc=[]):
    acc.append(x)
    return acc
print(bad_append(1))
print(bad_append(2))   # bất ngờ: [1, 2] chứ không phải [2]

[1]
[1, 2]


## 4. Cách 1 — Brute force: thử mọi cặp nút

Thu mọi giá trị vào list, so **mọi cặp** `i < j`, lấy `abs(a - b)` nhỏ nhất. Không dùng tính chất BST.

`float("inf")` là "vô cùng" — giá trị khởi tạo cho biến cần tìm **min** (mọi số đều nhỏ hơn nó).

In [6]:
def min_diff_bruteforce(root) -> int:
    vals = inorder_values(root)        # thứ tự không quan trọng ở cách này
    best = float("inf")
    for i in range(len(vals)):
        for j in range(i + 1, len(vals)):
            best = min(best, abs(vals[i] - vals[j]))
    return best

print(min_diff_bruteforce(root1), min_diff_bruteforce(root2))
print(float("inf") > 10**18, abs(3 - 7))

1 1
True 4


**Độ phức tạp:** `n` nút → `n(n-1)/2` cặp → `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n ≤ 100` vẫn chạy, nhưng không tận dụng BST.

## 5. Cách 2 — Sắp xếp rồi so phần tử liền kề

**Quan sát:** trong một dãy **đã sắp xếp** `a0 < a1 < ... < a(n-1)`, cặp có hiệu nhỏ nhất luôn là **hai phần tử liền kề**. Vì với `i < k < j`: `a[j] - a[i] = (a[j] - a[k]) + (a[k] - a[i]) >= a[k] - a[i]` → cặp "nhảy cóc" không bao giờ nhỏ hơn cặp liền kề bên trong nó.

→ Sắp xếp (`sorted`, `O(n log n)`) rồi xét `n-1` cặp liền kề.

In [7]:
vals = [6, 1, 4, 3, 2]
s = sorted(vals)
print("sorted:", s)
print("hiệu liền kề:", [s[i + 1] - s[i] for i in range(len(s) - 1)])

def min_diff_sort(root) -> int:
    s = sorted(inorder_values(root))
    return min(s[i + 1] - s[i] for i in range(len(s) - 1))

print(min_diff_sort(root1), min_diff_sort(root2))

sorted: [1, 2, 3, 4, 6]
hiệu liền kề: [1, 1, 1, 2]
1 1


### 5.1. `zip(s, s[1:])` — ghép cặp liền kề

`zip(a, b)` ghép phần tử cùng vị trí của hai dãy thành cặp. `zip(s, s[1:])` cho `(s0,s1), (s1,s2), ...` — cách Pythonic để duyệt cặp liền kề.

In [8]:
s = [1, 2, 3, 4, 6]
print(list(zip(s, s[1:])))
print(min(b - a for a, b in zip(s, s[1:])))

[(1, 2), (2, 3), (3, 4), (4, 6)]
1


## 6. Nút thắt và insight — BST đã "sắp xếp sẵn" qua inorder

Cách 2 tốn `O(n log n)` để sắp xếp. Nhưng với BST, **inorder đã cho dãy tăng dần** — không cần `sorted`! Hơn nữa ta không cần lưu cả list: chỉ cần nhớ **giá trị vừa thăm trước đó** (`prev`) và so với nút hiện tại.

```text
inorder:  1   2   3   4   6
prev:     -   1   2   3   4
hiệu:     -   1   1   1   2      -> min = 1
```

Bộ nhớ phụ chỉ còn `O(h)` cho đệ quy (`h` = chiều cao cây).

### 6.1. `nonlocal` — sửa biến của hàm bao ngoài

Hàm con `dfs` muốn **cập nhật** `prev` và `best` thuộc hàm ngoài. Nếu chỉ gán `prev = ...` trong `dfs`, Python tạo **biến cục bộ mới**. Khai báo `nonlocal prev, best` để nói "dùng biến của hàm bao ngoài".

In [9]:
def outer():
    counter = 0
    def inc():
        nonlocal counter
        counter += 1
    inc(); inc(); inc()
    return counter

print(outer())   # 3

def outer_wrong():
    counter = 0
    def inc():
        counter += 1      # UnboundLocalError nếu gọi: Python coi counter là biến cục bộ
    try:
        inc()
    except UnboundLocalError as e:
        print("Lỗi:", type(e).__name__)
    return counter

print(outer_wrong())

3
Lỗi: UnboundLocalError
0


## 7. Lời giải chính — inorder đệ quy + `prev`

```python
def min_diff_inorder(root):
    prev = None              # (1) giá trị nút vừa thăm trước (chưa có)
    best = float("inf")      # (2) đáp án tạm
    def dfs(node):
        nonlocal prev, best
        if node is None:     # (3) điều kiện dừng
            return
        dfs(node.left)       # (4) trái trước
        if prev is not None: # (5) "thăm" nút: so với nút liền trước trong dãy tăng
            best = min(best, node.val - prev)
        prev = node.val      # (6) nút hiện tại thành "trước" của nút sau
        dfs(node.right)      # (7) rồi phải
    dfs(root)
    return best
```
- `(5)`: dùng `prev is not None` chứ **không** `if prev:` — vì `prev` có thể là `0` (falsy)!
- `node.val - prev` luôn dương vì dãy tăng → không cần `abs`.

In [10]:
def min_diff_inorder(root) -> int:
    prev = None
    best = float("inf")

    def dfs(node):
        nonlocal prev, best
        if node is None:
            return
        dfs(node.left)
        if prev is not None:
            best = min(best, node.val - prev)
        prev = node.val
        dfs(node.right)

    dfs(root)
    return best

print(min_diff_inorder(root1), min_diff_inorder(root2))

1 1


## 8. TRACE từng bước

Ví dụ 2 (`[1,0,48,null,null,12,49]`), thứ tự thăm inorder: 0, 1, 12, 48, 49.

| Bước | node.val | prev trước | hiệu | best sau | prev sau |
|---|---|---|---|---|---|
| 1 | 0 | None | — | inf | 0 |
| 2 | 1 | 0 | 1 | 1 | 1 |
| 3 | 12 | 1 | 11 | 1 | 12 |
| 4 | 48 | 12 | 36 | 1 | 48 |
| 5 | 49 | 48 | 1 | 1 | 49 |

Hàm trace in thêm **độ sâu** để thấy đường đi đệ quy (vào trái, thăm, vào phải).

In [11]:
def trace_min_diff(root) -> int:
    prev, best, step = None, float("inf"), 0

    def dfs(node, depth):
        nonlocal prev, best, step
        if node is None:
            return
        print("  " * depth + f"vào nút {node.val} -> đi trái")
        dfs(node.left, depth + 1)
        step += 1
        if prev is not None:
            diff = node.val - prev
            new_best = min(best, diff)
            print("  " * depth + f"[bước {step}] THĂM {node.val}: prev={prev}, hiệu={diff}, best {best} -> {new_best}")
            best = new_best
        else:
            print("  " * depth + f"[bước {step}] THĂM {node.val}: prev=None (nút nhỏ nhất), chưa so")
        prev = node.val
        print("  " * depth + f"rời {node.val} -> đi phải")
        dfs(node.right, depth + 1)

    dfs(root, 0)
    print("Kết quả:", best)
    return best

_ = trace_min_diff(root2)

vào nút 1 -> đi trái
  vào nút 0 -> đi trái
  [bước 1] THĂM 0: prev=None (nút nhỏ nhất), chưa so
  rời 0 -> đi phải
[bước 2] THĂM 1: prev=0, hiệu=1, best inf -> 1
rời 1 -> đi phải
  vào nút 48 -> đi trái
    vào nút 12 -> đi trái
    [bước 3] THĂM 12: prev=1, hiệu=11, best 1 -> 1
    rời 12 -> đi phải
  [bước 4] THĂM 48: prev=12, hiệu=36, best 1 -> 1
  rời 48 -> đi phải
    vào nút 49 -> đi trái
    [bước 5] THĂM 49: prev=48, hiệu=1, best 1 -> 1
    rời 49 -> đi phải
Kết quả: 1


## 9. Code nộp LeetCode — lời giải chính

LeetCode đã định nghĩa sẵn `TreeNode`; ở đây ta dùng class tự định nghĩa ở mục 2 để chạy được trong notebook.

In [12]:
class Solution:
    def minDiffInBST(self, root: TreeNode | None) -> int:
        prev = None
        best = float("inf")

        def dfs(node):
            nonlocal prev, best
            if node is None:
                return
            dfs(node.left)
            if prev is not None:
                best = min(best, node.val - prev)
            prev = node.val
            dfs(node.right)

        dfs(root)
        return best

sol = Solution()
assert sol.minDiffInBST(root1) == 1
assert sol.minDiffInBST(root2) == 1
print("Solution OK")

Solution OK


## 10. Cách 4 — Inorder lặp bằng stack (không đệ quy)

Đệ quy dùng **call stack** ngầm. Ta có thể tự quản lý một **stack** (list với `append`/`pop` ở cuối — "vào sau ra trước", LIFO):

1. Đi trái hết mức, đẩy từng nút vào stack.
2. `pop` nút trên đỉnh → **thăm** nó (so với `prev`).
3. Chuyển sang cây con phải, lặp lại.

Ưu điểm: không lo giới hạn độ sâu đệ quy của Python (mặc định ~1000) khi cây rất lệch.

In [13]:
st = []
st.append(1); st.append(2); st.append(3)
print(st, "pop ->", st.pop(), "| còn", st)

def min_diff_iterative(root) -> int:
    stack, node = [], root
    prev, best = None, float("inf")
    while stack or node:
        while node:                 # 1. đi trái hết mức
            stack.append(node)
            node = node.left
        node = stack.pop()          # 2. thăm
        if prev is not None:
            best = min(best, node.val - prev)
        prev = node.val
        node = node.right           # 3. sang phải
    return best

print(min_diff_iterative(root1), min_diff_iterative(root2))

[1, 2] pop -> 3 | còn [1, 2]
1 1


In [14]:
class Solution:
    def minDiffInBST(self, root: TreeNode | None) -> int:
        stack, node = [], root
        prev, best = None, float("inf")
        while stack or node:
            while node:
                stack.append(node)
                node = node.left
            node = stack.pop()
            if prev is not None:
                best = min(best, node.val - prev)
            prev = node.val
            node = node.right
        return best

print(Solution().minDiffInBST(root1), Solution().minDiffInBST(root2))

1 1


## 11. Vì sao đúng?

**Bổ đề 1 (inorder BST tăng dần):** quy nạp theo cây: inorder(trái) gồm các giá trị `< root.val`, inorder(phải) gồm các giá trị `> root.val`, mỗi phần đã tăng dần ⇒ ghép `trái + [root] + phải` tăng dần.

**Bổ đề 2 (min nằm ở cặp liền kề):** đã chứng minh ở mục 5: với dãy tăng, mọi cặp không liền kề có hiệu ≥ một cặp liền kề nằm giữa chúng.

**Invariant khi duyệt:** sau khi thăm `k` nút đầu tiên theo inorder, `prev` = giá trị nút thứ `k`, và `best` = min hiệu của `k-1` cặp liền kề đầu tiên. Hết duyệt ⇒ `best` = min trên mọi cặp liền kề = min trên mọi cặp (bổ đề 2).

Vì `n >= 2`, có ít nhất một cặp → `best` không còn là `inf`.

## 12. Độ phức tạp và so sánh

`n` = số nút, `h` = chiều cao cây (`log n` nếu cân bằng, tới `n` nếu lệch).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Mọi cặp | `O(n²)` | `O(n)` | Không dùng BST |
| 2. Thu list + `sorted` | `O(n log n)` | `O(n)` | Đúng cho cây **bất kỳ** |
| 3. Inorder đệ quy + `prev` | `O(n)` | `O(h)` | **Khuyến nghị** |
| 4. Inorder lặp + stack | `O(n)` | `O(h)` | Tránh giới hạn đệ quy |

(Có thêm **Morris traversal** đạt `O(1)` bộ nhớ bằng cách tạm nối con trỏ — nâng cao, không cần cho bài Easy.)

## 13. Bộ test

1. Hai ví dụ đề.
2. Biên: cây 2 nút (con trái / con phải).
3. Cây lệch hẳn một phía (giống danh sách liên kết).
4. Có giá trị `0` (bẫy `if prev:`).
5. Min **không** nằm ở cặp cha–con (bẫy "chỉ so cha con").
6. Đối chiếu 4 cách trên BST ngẫu nhiên.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", min_diff_inorder(root1), 1)
check("đề 2", min_diff_inorder(root2), 1)
check("2 nút, con trái", min_diff_inorder(build_tree([10, 3])), 7)
check("2 nút, con phải", min_diff_inorder(build_tree([5, None, 100])), 95)
check("lệch phải", min_diff_inorder(build_tree([1, None, 5, None, 9, None, 20])), 4)
check("có giá trị 0", min_diff_inorder(build_tree([0, None, 3])), 3)
# min ở cặp KHÔNG phải cha-con: 48 (con phải của 30) và 50 (gốc)
tricky = build_tree([50, 30, 90, 10, 48])
print_tree(tricky)
check("không phải cha-con (48 và 50)", min_diff_inorder(tricky), 2)

PASS đề 1: 1
PASS đề 2: 1
PASS 2 nút, con trái: 7
PASS 2 nút, con phải: 95
PASS lệch phải: 4
PASS có giá trị 0: 3
    90
50
        48
    30
        10
PASS không phải cha-con (48 và 50): 2


In [16]:
import random
random.seed(783)

def insert_bst(root, val):
    if root is None:
        return TreeNode(val)
    if val < root.val:
        root.left = insert_bst(root.left, val)
    else:
        root.right = insert_bst(root.right, val)
    return root

for _ in range(300):
    values = random.sample(range(0, 10**5 + 1), random.randint(2, 100))   # giá trị khác nhau
    root = None
    for v in values:
        root = insert_bst(root, v)
    expected = min_diff_sort(root)
    assert inorder_values(root) == sorted(values)          # kiểm tra bổ đề 1
    assert min_diff_inorder(root) == expected
    assert min_diff_iterative(root) == expected
    assert Solution().minDiffInBST(root) == expected
    if len(values) <= 30:
        assert min_diff_bruteforce(root) == expected
print("300 BST ngẫu nhiên: các cách khớp nhau, inorder luôn tăng dần")

300 BST ngẫu nhiên: các cách khớp nhau, inorder luôn tăng dần


## 14. Lỗi người mới hay gặp

### Lỗi 1: Chỉ so cha với con
```python
best = min(best, abs(node.val - node.left.val), abs(node.val - node.right.val))  # SAI
```
Cặp gần nhất có thể ở xa nhau trong cây (ví dụ 48 và 50 ở trên).

### Lỗi 2: `if prev:` thay vì `if prev is not None:`
Khi `prev = 0`, `if prev:` là `False` → bỏ qua một so sánh.

### Lỗi 3: Quên `nonlocal`
Gán `prev = node.val` trong hàm con mà không `nonlocal` → `UnboundLocalError` hoặc `prev` ngoài không bao giờ đổi.

### Lỗi 4: Duyệt preorder (gốc → trái → phải) rồi so liền kề
Preorder **không** cho dãy tăng → so cặp liền kề sai.

In [17]:
def wrong_parent_child(root):
    best = float("inf")
    def dfs(node):
        nonlocal best
        if node is None:
            return
        for child in (node.left, node.right):
            if child:
                best = min(best, abs(node.val - child.val))
        dfs(node.left); dfs(node.right)
    dfs(root)
    return best

def wrong_truthy_prev(root):
    prev, best = None, float("inf")
    def dfs(node):
        nonlocal prev, best
        if node is None:
            return
        dfs(node.left)
        if prev:                       # SAI khi prev == 0
            best = min(best, node.val - prev)
        prev = node.val
        dfs(node.right)
    dfs(root)
    return best

print("tricky: cha-con =", wrong_parent_child(tricky), "| đúng =", min_diff_inorder(tricky))
t0 = build_tree([0, None, 3])
print("[0,null,3]: if prev: =", wrong_truthy_prev(t0), "| đúng =", min_diff_inorder(t0))

tricky: cha-con = 18 | đúng = 2
[0,null,3]: if prev: = inf | đúng = 3


## 15. Mẹo nhận dạng pattern

- Đề có **BST** + hỏi gì đó về **thứ tự / hiệu / phần tử thứ k** ⇒ nghĩ ngay tới **inorder = dãy tăng**.
- "Hiệu nhỏ nhất giữa hai phần tử bất kỳ" ⇒ **sắp xếp** rồi xét **liền kề**.
- Cần thông tin "phần tử trước đó" khi duyệt ⇒ giữ biến `prev` (với đệ quy dùng `nonlocal` hoặc biến instance `self.prev`).
- Cây có thể sâu ⇒ cân nhắc phiên bản **stack** lặp.

### Biến thể

1. **LeetCode 530** (giống hệt).
2. **Kth Smallest Element in a BST (230)**: inorder, dừng ở phần tử thứ `k`.
3. **Validate BST (98)**: inorder phải **tăng nghiêm ngặt** — cũng dùng `prev`.
4. **Hiệu lớn nhất** giữa hai nút: chỉ cần `max - min` = nút phải nhất − nút trái nhất.
5. Cây nhị phân **thường** (không BST): phải thu giá trị và `sorted` (cách 2).

In [18]:
# Biến thể 4: hiệu lớn nhất trong BST = phải nhất - trái nhất
def max_diff_bst(root):
    lo = hi = root
    while lo.left:
        lo = lo.left
    while hi.right:
        hi = hi.right
    return hi.val - lo.val

print(max_diff_bst(root1), max_diff_bst(root2))

5 49


## 16. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (tính tay):** Cây `[27, None, 34, None, 58, 50, None, 44]`. Viết dãy inorder và đáp án `minDiffInBST`. Gán `answer_ex1 = (list_inorder, đáp_án)`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `inorder_exercise(root)` trả về list inorder (đệ quy).

**Bài 3 (viết lại):** `kth_smallest(root, k)` trả về giá trị nhỏ thứ `k` (1-indexed) dùng inorder lặp bằng stack, dừng sớm.

**Bài 4 (biến thể):** `is_valid_bst(root)` kiểm tra cây có phải BST (giá trị tăng **nghiêm ngặt** theo inorder) dùng `prev`.

**Bài 5 (mini challenge):** `min_diff_any_tree(root)` cho cây nhị phân **bất kỳ** (không đảm bảo BST).

In [19]:
answer_ex1 = None   # TODO: ([...], số)

def inorder_exercise(root) -> list[int]:
    # TODO
    pass

def kth_smallest(root, k: int) -> int:
    # TODO: stack + đếm số nút đã thăm
    pass

# assert inorder_exercise(root1) == [1, 2, 3, 4, 6]
# assert kth_smallest(root2, 3) == 12

In [20]:
def is_valid_bst(root) -> bool:
    # TODO: inorder, nếu node.val <= prev -> False
    pass

def min_diff_any_tree(root) -> int:
    # TODO
    pass

# assert is_valid_bst(root1) == True
# assert is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])) == False
# assert min_diff_any_tree(build_tree([1, 50, 2])) == 1

### Gợi ý 3 mức

**Bài 1** — (1) Vẽ cây: 27 có con phải 34, 34 có con phải 58, 58 có con trái 50, 50 có con trái 44. (2) Inorder = đi trái trước. (3) Dãy là 27, 34, 44, 50, 58.

**Bài 2** — (1) Trái, gốc, phải. (2) Trả list rỗng nếu `None`. (3) `return inorder(left) + [root.val] + inorder(right)`.

**Bài 3** — (1) Dùng template stack ở mục 10. (2) Mỗi lần `pop` là thăm một nút → tăng bộ đếm. (3) Bộ đếm bằng `k` thì `return node.val`.

**Bài 4** — (1) Inorder của BST tăng nghiêm ngặt. (2) Giữ `prev`. (3) `if prev is not None and node.val <= prev: return False`.

**Bài 5** — (1) Không có thứ tự sẵn. (2) Thu mọi giá trị (duyệt kiểu nào cũng được). (3) `sorted` rồi so liền kề.

---

## 17. Đáp án tham khảo

In [21]:
answer_ex1_solution = ([27, 34, 44, 50, 58], 6)

def inorder_exercise_solution(root):
    if root is None:
        return []
    return inorder_exercise_solution(root.left) + [root.val] + inorder_exercise_solution(root.right)

def kth_smallest_solution(root, k):
    stack, node, seen = [], root, 0
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        seen += 1
        if seen == k:
            return node.val
        node = node.right
    raise ValueError("k lớn hơn số nút")

def is_valid_bst_solution(root):
    stack, node, prev = [], root, None
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        if prev is not None and node.val <= prev:
            return False
        prev = node.val
        node = node.right
    return True

def min_diff_any_tree_solution(root):
    vals, stack = [], [root]
    while stack:
        node = stack.pop()
        if node:
            vals.append(node.val)
            stack.append(node.left)
            stack.append(node.right)
    vals.sort()
    return min(b - a for a, b in zip(vals, vals[1:]))

t_ex = build_tree([27, None, 34, None, 58, 50, None, 44])
print_tree(t_ex)
assert (inorder_values(t_ex), min_diff_inorder(t_ex)) == answer_ex1_solution
assert inorder_exercise_solution(root1) == [1, 2, 3, 4, 6]
assert kth_smallest_solution(root2, 3) == 12 and kth_smallest_solution(root1, 1) == 1
assert is_valid_bst_solution(root1) is True
assert is_valid_bst_solution(build_tree([5, 1, 4, None, None, 3, 6])) is False
assert min_diff_any_tree_solution(build_tree([1, 50, 2])) == 1
print("Đáp án PASS toàn bộ test!")

        58
            50
                44
    34
27
Đáp án PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def minDiffInBST(self, root: TreeNode | None) -> int:
        prev, best = None, float("inf")
        def dfs(node):
            nonlocal prev, best
            if not node:
                return
            dfs(node.left)
            if prev is not None:
                best = min(best, node.val - prev)
            prev = node.val
            dfs(node.right)
        dfs(root)
        return best
```

Công thức nhớ: **BST + inorder = dãy tăng; min hiệu = min cặp liền kề.**

**Checklist trước khi submit:**
- [ ] Duyệt **inorder** (trái → gốc → phải), không phải preorder?
- [ ] `prev is not None` (không phải `if prev:`) vì giá trị có thể là 0?
- [ ] Đã khai báo `nonlocal prev, best`?
- [ ] Khởi tạo `best = float("inf")`; hàm trả `int` (vì có ≥ 2 nút nên `best` luôn được cập nhật)?
- [ ] Đã thử cây mà cặp gần nhất **không** là cha–con?